# Day 15 - Custom Transformers and Feature Scaling

This notebook explains how to build reusable preprocessing steps and why scaling numeric features is important for many machine learning algorithms.

## Learning Goals
- Understand what a custom transformer is and why it is useful.
- Learn how `StandardScaler`, `MinMaxScaler`, and `RobustScaler` change numeric data.
- Build a pipeline that combines feature creation and scaling.

> Good preprocessing helps models work better by keeping feature values in the right range and avoiding duplicate code.

## Why preprocessing matters
Machine learning models often expect numeric inputs on a similar scale.
If one feature is much larger than another, it can dominate the model, even when it is not more important.

Creating a custom transformer also helps keep your code clean and reusable, especially when you need the same transformation during training and testing.

In [1]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.pipeline import Pipeline

# Create a small dataset with height and weight.
# Height is in centimeters and weight is in kilograms.
# These two features are on different scales, so scaling helps.
df = pd.DataFrame({
    'height': [150, 160, 170, 180, 190],
    'weight': [50, 60, 72, 85, 95]
})

df

,height,weight
0,150,50
1,160,60
2,170,72
3,180,85
4,190,95


## Example dataset explanation
- `height` is measured in centimeters.
- `weight` is measured in kilograms.

Because `height` values range from 150 to 190 and `weight` values range from 50 to 95, models can become biased toward the larger-scale feature unless we scale the data.

## Custom Transformer: adding BMI
BMI is a useful derived feature when we have height and weight.
A custom transformer is a class with `fit()` and `transform()` methods.

- `fit()` learns from data if needed (for example, computing means or medians).
- `transform()` returns the modified data.

This pattern is compatible with scikit-learn pipelines.

In [2]:
class BMITransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # No learning is needed for BMI, so we just return self.
        return self

    def transform(self, X):
        # Work with a copy to avoid changing the original input.
        X = X.copy()

        # Convert height from centimeters to meters for BMI calculation.
        height_meters = X['height'] / 100
                                                                                                                                                      
        # BMI = weight / (height in meters)^2
        X['BMI'] = X['weight'] / (height_meters * height_meters)
        return X

bmi_transformer = BMITransformer()
transformed_df = bmi_transformer.transform(df)
transformed_df

,height,weight,BMI
0,150,50,22.222222
1,160,60,23.437500
2,170,72,24.913495
3,180,85,26.234568
4,190,95,26.315789


## Why BMI as a custom transformer?
- It creates a new feature from existing ones.
- It is reusable in training and inference.
- It keeps transformation logic separate from model code.

In a real project, custom transformers can compute many different features from raw input data.

## Feature Scaling: what and why
Feature scaling changes numeric values so they are easier for models to use.

- `StandardScaler` centers data to mean 0 and scales to unit variance.
- `MinMaxScaler` rescales values to the range [0, 1].
- `RobustScaler` uses percentiles and is less sensitive to outliers.

Use scaling when features have different units or ranges, especially for distance-based algorithms like KNN or gradient-based models like linear regression.

In [3]:
X = df[['height', 'weight']]

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

for name, scaler in scalers.items():
    transformed = scaler.fit_transform(X)
    transformed_df = pd.DataFrame(transformed, columns=X.columns)
    print(f"{name} result:")
    print(transformed_df)
    print()

StandardScaler result:
     height    weight
0 -1.414214 -1.375917
1 -0.707107 -0.761668
2  0.000000 -0.024570
3  0.707107  0.773953
4  1.414214  1.388202

MinMaxScaler result:
   height    weight
0    0.00  0.000000
1    0.25  0.222222
2    0.50  0.488889
3    0.75  0.777778
4    1.00  1.000000

RobustScaler result:
   height  weight
0    -1.0   -0.88
1    -0.5   -0.48
2     0.0    0.00
3     0.5    0.52
4     1.0    0.92



## Interpreting the scaler outputs
- `StandardScaler` output has values around 0, because it subtracts the mean and divides by standard deviation.
- `MinMaxScaler` output maps the smallest value to 0 and the largest to 1.
- `RobustScaler` uses the median and interquartile range, so extreme values have less effect.

This is why `RobustScaler` is often a good choice when the dataset may contain outliers.

## Pipeline: combining transformations
A pipeline chains preprocessing steps so you can run them in one call.
Here we first compute BMI, then scale all numeric columns.

This is better than writing separate code for each step because it reduces errors and makes the workflow reproducible.

In [4]:
pipe = Pipeline([
    ('bmi', BMITransformer()),
    ('scale', StandardScaler())
])

scaled_result = pipe.fit_transform(df)
# The pipeline now returns height, weight, and BMI, so provide three column names.
scaled_df = pd.DataFrame(scaled_result, columns=['height', 'weight', 'BMI'])
scaled_df

,height,weight,BMI
0,-1.414214,-1.375917,-1.506497
1,-0.707107,-0.761668,-0.744450
2,0.000000,-0.024570,0.181081
3,0.707107,0.773953,1.009468
4,1.414214,1.388202,1.060398


## What is happening in the pipeline?
- `BMITransformer()` adds a new column called `BMI`.
- `StandardScaler()` scales all numeric columns, including `height`, `weight`, and `BMI`.
- `fit_transform()` applies the full preprocessing sequence in one step.

If you later use the same pipeline on new data, it will apply the exact same transformations automatically.

## Summary and practice
- Custom transformers are reusable preprocessing classes with `fit()` and `transform()` methods.
- Scaling keeps feature values on a similar range and improves many machine learning algorithms.
- `StandardScaler` is a good default for normally distributed data.
- `MinMaxScaler` is useful when you want values between 0 and 1.
- `RobustScaler` is better when your data may contain outliers.

Try these exercises:
- Add a new feature such as `age` and scale it too.
- Change the pipeline to use `MinMaxScaler` instead of `StandardScaler`.
- Create another transformer that computes `weight / height` and compare it with `BMI`.
- Add an outlier to `weight` (for example, 200 kg) and see how each scaler behaves.